In [13]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from pycaret.classification import *
import lightgbm as lgb
import optuna
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, accuracy_score


# 学習用データの読み込み
train_data = pd.read_csv('../data/train.csv')
test_data = pd.read_csv('../data/test.csv')

# 列名の確認
print("Original train data columns:", train_data.columns)
print("Original test data columns:", test_data.columns)

Original train data columns: Index(['index', 'Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
       'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'],
      dtype='object')
Original test data columns: Index(['index', 'Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
       'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age'],
      dtype='object')


In [14]:

# 前処理（異常値除去と欠損値補完）
train_data = train_data.drop(train_data[train_data['BloodPressure'] < 20].index)
train_data = train_data.drop(train_data[train_data['BMI'] < 10].index)

train_data['SkinThickness'] = train_data['SkinThickness'].replace(0, np.nan)
train_data['Insulin'] = train_data['Insulin'].replace(0, np.nan)
test_data['SkinThickness'] = test_data['SkinThickness'].replace(0, np.nan)
test_data['Insulin'] = test_data['Insulin'].replace(0, np.nan)

# KNN Imputer のインスタンス作成
imputer = KNNImputer(n_neighbors=5, weights='distance', metric='nan_euclidean')

# 訓練データから'index'と'Outcome'を除いた部分でfit_transform
X_train_for_impute = train_data.drop(columns=['index', 'Outcome'])
train_data_imputed = imputer.fit_transform(X_train_for_impute)

# テストデータの'index'を保持しつつ、それ以外でtransform
test_index = test_data['index']  # 'index'列を分離
X_test_for_impute = test_data.drop(columns=['index'])
test_data_imputed = imputer.transform(X_test_for_impute)

# NumPy配列からDataFrameに戻す
train_data_imputed_df = pd.DataFrame(train_data_imputed, columns=X_train_for_impute.columns)
train_data = pd.concat([train_data['index'].reset_index(drop=True), train_data_imputed_df, train_data['Outcome'].reset_index(drop=True)], axis=1)
test_data = pd.concat([test_index.reset_index(drop=True), pd.DataFrame(test_data_imputed, columns=X_test_for_impute.columns)], axis=1)

# データの形状を確認
print("Train data shape after preprocessing:", train_data.shape)
print("Test data shape after preprocessing:", test_data.shape)

# 結果の表示
print("\n補完後のデータ:")
print(train_data.head(15))
print(test_data.head(15))

# # 散布図の作成
# x_column = 'Outcome'
# for column in train_data.columns:
#     if column != x_column and column != 'index':
#         plt.figure(figsize=(10, 6))
#         plt.scatter(train_data[x_column], train_data[column], label=column)
#         plt.xlabel(x_column)
#         plt.ylabel(column)
#         plt.title(f'Scatter Plot: {x_column} vs {column}')
#         plt.legend()
#         plt.grid(True, alpha=0.3)
#         plt.show()

# # ヒストグラムの作成
# for column in train_data.columns:
#   if column != x_column and column != 'index':
#     plt.figure(figsize=(10, 6))  # 新しい図を作成
#     plt.hist(train_data[column], bins=50)
#     plt.title(f'Histogram of {column}')
#     plt.xlabel('Value')
#     plt.ylabel('Frequency')
#     plt.show()

Train data shape after preprocessing: (2746, 10)
Test data shape after preprocessing: (1919, 9)

補完後のデータ:
    index  Pregnancies  Glucose  BloodPressure  SkinThickness     Insulin  \
0     200          9.0    125.0           74.0      34.362578  111.502492   
1    3832          4.0    109.0           80.0      16.857819  132.688253   
2    4927          4.0     88.0           78.0      39.000000  122.593425   
3    4088          9.0    125.0           74.0      32.499902  118.972197   
4    3644          5.0    107.0           78.0      44.000000  284.000000   
5    3323          5.0     84.0           64.0      19.423263  134.150895   
6    2203          2.0    138.0           86.0      30.000000   92.284713   
7     565          1.0    123.0           52.0      32.378424  119.311096   
8    1386          1.0    138.0           76.0      30.000000   78.426243   
9    4162          7.0    100.0           74.0      29.593793  164.996909   
10   4818          9.0    125.0           76.0 

In [18]:
# PyCaret 3.xのセットアップ
clf_setup = setup(
    data=train_data,
    target='Outcome',
    ignore_features=['index'],  # 'index'列を無視
    polynomial_features=True,
    bin_numeric_features=True,
    transformation=True,
    transformation_method='yeo-johnson',
    normalize=True,
    feature_selection=True,  # 特徴量選択を有効化
    remove_multicollinearity=True,  # 多重共線性の除去
    multicollinearity_threshold=0.85,  # 多重共線性の閾値
    preprocess=True,
    use_gpu=True  # GPUを使用（利用可能な場合）
)

# 変換後の訓練データを取得
transformed_train = get_config('X_train_transformed')  # 3.xでは'_transformed'付き
y_train = get_config('y_transformed')  # 変換後の目的変数

# 前処理パイプラインを取得してテストデータを変換
pipeline = get_config('pipeline')
transformed_test = pipeline.transform(test_data.drop(columns=['index']))

# index列を保持
train_data_transformed = pd.concat([train_data['index'], transformed_train.reset_index(drop=True)], axis=1)
train_data_transformed['Outcome'] = y_train.values
test_data_transformed = pd.concat([test_data['index'].reset_index(drop=True), pd.DataFrame(transformed_test, columns=transformed_train.columns).reset_index(drop=True)], axis=1)

# データの形状を確認
print("Transformed train data shape:", train_data_transformed.shape)
print("Transformed test data shape:", test_data_transformed.shape)

# データ保存
train_data_transformed.to_csv('../data/train_pycaret_transformed.csv', index=False)
test_data_transformed.to_csv('../data/test_pycaret_transformed.csv', index=False)

TypeError: 'bool' object is not iterable

In [5]:
# 特殊文字をアンダースコアに置換
train_data_transformed.columns = train_data_transformed.columns.str.replace(r'[{}\[\]":]', '_', regex=True)
test_data_transformed.columns = test_data_transformed.columns.str.replace(r'[{}\[\]":]', '_', regex=True)

# データ準備
X = train_data_transformed.drop(columns=['index', 'Outcome'])
y = train_data_transformed['Outcome']
X_test = test_data_transformed.drop(columns=['index'])

# 訓練データとテストデータに分割（ハイパーパラメータ調整用）
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [6]:
# Optunaによるハイパーパラメータ最適化
def objective(trial):
    params = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'num_leaves': trial.suggest_int('num_leaves', 20, 50),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.7, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.7, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 20),
        'lambda_l1': trial.suggest_float('lambda_l1', 0.0, 1.0),
        'lambda_l2': trial.suggest_float('lambda_l2', 0.0, 1.0),
        'verbose': -1
    }

    # 交差検証
    k = 10
    kf = KFold(n_splits=k, shuffle=True, random_state=42)
    auc_scores = []

    for train_idx, val_idx in kf.split(X):
        X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
        y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]

        lgb_train = lgb.Dataset(X_train_fold, y_train_fold)
        lgb_val = lgb.Dataset(X_val_fold, y_val_fold, reference=lgb_train)

        gbm = lgb.train(
            params,
            lgb_train,
            num_boost_round=100,
            valid_sets=[lgb_val],
            callbacks=[lgb.early_stopping(stopping_rounds=10, verbose=False)]
        )

        y_pred_proba = gbm.predict(X_val_fold)
        auc = roc_auc_score(y_val_fold, y_pred_proba)
        auc_scores.append(auc)

    return np.mean(auc_scores)

# Optunaで最適化
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

# 最適なパラメータ
print("最適なハイパーパラメータ:", study.best_params)
print("最適なAUC:", study.best_value)

# 最適パラメータで最終モデル訓練
best_params = study.best_params
best_params['objective'] = 'binary'
best_params['metric'] = 'auc'
best_params['verbose'] = -1

lgb_train = lgb.Dataset(X, y)
gbm = lgb.train(
    best_params,
    lgb_train,
    num_boost_round=100,
    valid_sets=[lgb_train],
    callbacks=[lgb.early_stopping(stopping_rounds=10)]
)

# 交差検証で性能評価
k = 10
kf = KFold(n_splits=k, shuffle=True, random_state=42)
metrics = {'accuracy': [], 'auc': [], 'precision': [], 'recall': [], 'f1': []}

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    print(f"\nFold {fold + 1}/{k}")
    X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
    y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]

    lgb_train = lgb.Dataset(X_train_fold, y_train_fold)
    gbm = lgb.train(
        best_params,
        lgb_train,
        num_boost_round=100,
        valid_sets=[lgb_train],
        callbacks=[lgb.early_stopping(stopping_rounds=10)]
    )
    y_pred_proba = gbm.predict(X_val_fold)
    y_pred = [1 if x > 0.5 else 0 for x in y_pred_proba]

    metrics['accuracy'].append(accuracy_score(y_val_fold, y_pred))
    metrics['auc'].append(roc_auc_score(y_val_fold, y_pred_proba))
    metrics['precision'].append(precision_score(y_val_fold, y_pred))
    metrics['recall'].append(recall_score(y_val_fold, y_pred))
    metrics['f1'].append(f1_score(y_val_fold, y_pred))

# 評価結果の表示
for metric, scores in metrics.items():
    print(f"{metric.capitalize()}: {np.mean(scores):.4f} (±{np.std(scores):.4f})")

# 特徴量の重要度
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': gbm.feature_importance(importance_type='gain')
})
print("\n特徴量の重要度:")
print(feature_importance.sort_values(by='Importance', ascending=False))

最適なハイパーパラメータ: {'num_leaves': 28, 'learning_rate': 0.013903242134803867, 'feature_fraction': 0.7788878229960683, 'bagging_fraction': 0.9817426344848754, 'bagging_freq': 3, 'min_child_samples': 7, 'lambda_l1': 0.5095716905403893, 'lambda_l2': 0.2779067244955239}
最適なAUC: 0.6469561910895669

Fold 1/10

Fold 2/10

Fold 3/10

Fold 4/10

Fold 5/10

Fold 6/10

Fold 7/10

Fold 8/10

Fold 9/10

Fold 10/10
Accuracy: 0.7724 (±0.0093)
Auc: 0.6285 (±0.0271)
Precision: 0.6211 (±0.0705)
Recall: 0.1141 (±0.0317)
F1: 0.1914 (±0.0466)

特徴量の重要度:
  Feature   Importance
0     BMI  8643.346624


In [7]:
# テストデータで予測
y_pred_proba = gbm.predict(X_test)
y_pred = [1 if x > 0.5 else 0 for x in y_pred_proba]

# 出力用データフレーム
output = pd.DataFrame({
    'index': test_data_transformed['index'],
    'Outcome': y_pred
})

# CSVに保存
output.to_csv('../data/prediction_pycaret.csv', header=False, index=False)
print("予測結果を 'prediction_pycaret.csv' に出力しました。")

予測結果を 'prediction_pycaret.csv' に出力しました。
